# 14 -- Batch Scoring Pipeline
**Purpose:** Build the end-to-end scoring pipeline: transactions.csv in, scored_transactions.csv out.

**Input:** `data/raw/transactions.csv` (or any new CSV in the same schema)
**Output:** `data/output/scored_transactions.csv` | `data/output/audit_log.jsonl`

## Learning Objectives
- Understand what a batch scoring pipeline is and how it differs from real-time
- Apply the full preprocessing stack (clean -> features -> scale -> score) in one function
- Generate the PMLA-compliant JSONL audit log
- Apply risk tier thresholds from config/threshold.yaml

## Output Schema (PoC Plan V5 -- Section 7)
scored_transactions.csv columns:
transaction_id | risk_score | risk_tier | scored_at | model_version |
f1_name | f1_value | f1_contribution |
f2_name | f2_value | f2_contribution |
f3_name | f3_value | f3_contribution |
f4_name | f4_value | f4_contribution |
f5_name | f5_value | f5_contribution

## PMLA Audit Log (JSONL)
Each line: {"transaction_id": ..., "risk_score": ..., "risk_tier": ...,
"model_version": ..., "scored_at": ..., "top_features": [...]}

## Step 1 -- Build the Preprocessing Function
Write `preprocess(df)` that applies: cleaning -> feature engineering -> scaling.
This must be identical to what was applied in Notebooks 03, 04, 05.

## Step 2 -- Build the Scoring Function
Write `score(df, model, threshold_config)` that returns risk_score, risk_tier, and top-5 SHAP features for each transaction.

## Step 3 -- Build the Pipeline
Write `run_batch_scoring(input_path, output_path, audit_log_path)` that:
1. Loads the CSV
2. Calls preprocess()
3. Calls score()
4. Writes scored_transactions.csv
5. Writes audit_log.jsonl

## Step 4 -- Run and Validate
Run the pipeline on the full V3.1 dataset.
Verify: every transaction has a risk_score and risk_tier. Audit log has 50,800 lines. No NaN in output.

## Definition of Done
- [ ] preprocess(), score(), run_batch_scoring() all written and tested
- [ ] scored_transactions.csv matches output schema from PoC Plan V5
- [ ] audit_log.jsonl generated with one JSON object per transaction
- [ ] Risk tier distribution printed (GREEN / AMBER / RED counts)
- [ ] Pipeline runs on a fresh CSV with no modifications needed

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import shap
from datetime import datetime, timezone

# Load feature matrix (all 50,800 transactions)
X = pd.read_parquet(r"../data/processed\feature_matrix_clean.parquet")

# Load champion model
model = joblib.load(r"../data/processed\xgboost_champion.pkl")

# Load optimal threshold
with open(r"../data/processed\threshold_config.json") as f:
    threshold_config = json.load(f)
threshold = threshold_config["optimal_threshold"]

print(f"Transactions to score: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Threshold: {threshold}")


Transactions to score: 50800
Features: 46
Threshold: 0.9217


In [2]:
# Score all transactions
y_prob = model.predict_proba(X)[:, 1]

# Convert probability to 0-100 risk score
risk_scores = (y_prob * 100).round(2)

# Assign risk tier
def assign_tier(score):
    if score >= 80:
        return "RED"
    elif score >= 40:
        return "AMBER"
    else:
        return "GREEN"

risk_tiers = [assign_tier(s) for s in risk_scores]

print(f"RED   (high risk):    {risk_tiers.count('RED')}")
print(f"AMBER (medium risk):  {risk_tiers.count('AMBER')}")
print(f"GREEN (low risk):     {risk_tiers.count('GREEN')}")


RED   (high risk):    242
AMBER (medium risk):  2
GREEN (low risk):     50556


In [3]:
import xgboost as xgb

booster = model.get_booster()
dmatrix = xgb.DMatrix(X)

# Compute SHAP values for all transactions
shap_matrix = booster.predict(dmatrix, pred_contribs=True)
shap_matrix = shap_matrix[:, :-1]  # drop bias column

feature_names = X.columns.tolist()
print(f"SHAP matrix shape: {shap_matrix.shape}")


SHAP matrix shape: (50800, 46)


In [4]:
from datetime import datetime, timezone

scored_at = datetime.now(timezone.utc).isoformat()
model_version = "xgboost-2.1.3-v1"

rows = []

for i in range(len(X)):
    # Get top 5 features by absolute SHAP value
    shap_row = shap_matrix[i]
    top5_idx = np.argsort(np.abs(shap_row))[::-1][:5]

    row = {
        "transaction_id": i,
        "risk_score": round(float(risk_scores[i]), 2),
        "risk_tier": risk_tiers[i],
        "scored_at": scored_at,
        "model_version": model_version,
        "f1_name": feature_names[top5_idx[0]],
        "f2_name": feature_names[top5_idx[1]],
        "f3_name": feature_names[top5_idx[2]],
        "f4_name": feature_names[top5_idx[3]],
        "f5_name": feature_names[top5_idx[4]],
        "f1_value": round(float(X.iloc[i, top5_idx[0]]), 4),
        "f2_value": round(float(X.iloc[i, top5_idx[1]]), 4),
        "f3_value": round(float(X.iloc[i, top5_idx[2]]), 4),
        "f4_value": round(float(X.iloc[i, top5_idx[3]]), 4),
        "f5_value": round(float(X.iloc[i, top5_idx[4]]), 4),
        "f1_contribution": round(float(shap_row[top5_idx[0]]), 4),
        "f2_contribution": round(float(shap_row[top5_idx[1]]), 4),
        "f3_contribution": round(float(shap_row[top5_idx[2]]), 4),
        "f4_contribution": round(float(shap_row[top5_idx[3]]), 4),
        "f5_contribution": round(float(shap_row[top5_idx[4]]), 4),
    }
    rows.append(row)

scored_df = pd.DataFrame(rows)
print(f"Scored transactions: {len(scored_df)}")
print(scored_df[scored_df["risk_tier"] == "RED"].head(3)[
    ["transaction_id","risk_score","risk_tier","f1_name","f1_contribution"]
])


Scored transactions: 50800
     transaction_id  risk_score risk_tier             f1_name  f1_contribution
243             243       99.94       RED  is_primary_account           3.4947
521             521       99.95       RED     is_round_number          11.0936
739             739       99.90       RED   amount_zscore_90d           4.0068


In [5]:
# Save scored transactions CSV
output_path = r"../data/processed\scored_transactions.csv"
scored_df.to_csv(output_path, index=False)
print(f"Saved: scored_transactions.csv ({len(scored_df)} rows)")

# Save PMLA audit log — one JSON line per RED transaction
audit_path = r"../data/processed\audit_log.jsonl"
red_df = scored_df[scored_df["risk_tier"] == "RED"]

with open(audit_path, "w") as f:
    for _, row in red_df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")

print(f"Saved: audit_log.jsonl ({len(red_df)} RED transactions logged)")


Saved: scored_transactions.csv (50800 rows)
Saved: audit_log.jsonl (242 RED transactions logged)
